In [2]:
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="ticks", font_scale=1.05)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

print("Biomarker analysis environment initialized.")

Biomarker analysis environment initialized.


In [3]:
# 1. Load cell line annotations and CDK7 dependency scores
meta_df = pd.read_csv("../data/raw/Model.csv", low_memory=False)
id_col = 'DepMap_ID' if 'DepMap_ID' in meta_df.columns else meta_df.columns[0]
meta_df = meta_df.rename(columns={id_col: 'ModelID'})

crispr_path = "../data/raw/CRISPRGeneEffect.csv"
header_df = pd.read_csv(crispr_path, nrows=0)
cdk7_col = [c for c in header_df.columns if "CDK7" in c][0]
first_col = header_df.columns[0]

cdk7_df = pd.read_csv(crispr_path, usecols=[first_col, cdk7_col])
cdk7_df = cdk7_df.rename(columns={first_col: 'ModelID', cdk7_col: 'CDK7_Chronos_Score'})

df = pd.merge(cdk7_df, meta_df[['ModelID', 'lineage', 'primary_disease']], on='ModelID').dropna(subset=['CDK7_Chronos_Score'])

# 2. Stratify cohorts into Sensitive (Top 25%) and Resistant (Bottom 25%)
q25 = df['CDK7_Chronos_Score'].quantile(0.25)
q75 = df['CDK7_Chronos_Score'].quantile(0.75)

df['Response_Group'] = 'Intermediate'
df.loc[df['CDK7_Chronos_Score'] >= q75, 'Response_Group'] = 'Sensitive'
df.loc[df['CDK7_Chronos_Score'] <= q25, 'Response_Group'] = 'Resistant'

print(f"Resistant threshold (score <= Q25): {q25:.4f} (n = {(df['Response_Group'] == 'Resistant').sum()})")
print(f"Sensitive threshold (score >= Q75): {q75:.4f} (n = {(df['Response_Group'] == 'Sensitive').sum()})")

Resistant threshold (score <= Q25): 0.9160 (n = 442)
Sensitive threshold (score >= Q75): 1.0443 (n = 442)


In [4]:
import os

print("Files in raw data folder:")
print(os.listdir("../data/raw"))

Files in raw data folder:
['CRISPRGeneEffect.csv', 'OmicsExpressionProteinCodingGenesTPMLogp1.csv', 'Model.csv']


In [5]:
expr_path = "../data/raw/OmicsExpressionProteinCodingGenesTPMLogp1.csv"
expr_header = pd.read_csv(expr_path, nrows=0)
expr_first_col = expr_header.columns[0]

# List of hypothesized target genes
target_symbols = [
    'CDK7', 'MYC', 'MCL1', 'BCL2', 'BCL2L1', 
    'CDK9', 'CDK12', 'CDK13', 'CCNT1', 
    'ZEB1', 'VIM', 'CDH1', 'ABCB1', 'ABCG2'
]

# Map gene symbols to DepMap column headers (format: "SYMBOL (ENTREZ_ID)")
matched_cols = {}
for sym in target_symbols:
    matches = [c for c in expr_header.columns if c.startswith(f"{sym} ")]
    if matches:
        matched_cols[sym] = matches[0]

print(f"Found {len(matched_cols)}/{len(target_symbols)} candidate biomarker columns.")

# Extract only relevant expression columns
usecols = [expr_first_col] + list(matched_cols.values())
expr_df = pd.read_csv(expr_path, usecols=usecols)
expr_df = expr_df.rename(columns={expr_first_col: 'ModelID'})

# Clean column headers back to simple gene symbols
rename_dict = {v: k for k, v in matched_cols.items()}
expr_df = expr_df.rename(columns=rename_dict)

# Merge with stratification groups
analysis_df = pd.merge(df, expr_df, on='ModelID', how='inner')
analysis_df = analysis_df[analysis_df['Response_Group'].isin(['Sensitive', 'Resistant'])].copy()

print(f"Cell lines with matched CRISPR + Expression data: {len(analysis_df)}")

Found 14/14 candidate biomarker columns.
Cell lines with matched CRISPR + Expression data: 716


In [6]:
# 1. Inspect the available gene columns in CRISPRGeneEffect.csv
crispr_path = "../data/raw/CRISPRGeneEffect.csv"
header_df = pd.read_csv(crispr_path, nrows=0)
first_col = header_df.columns[0]

# Candidate resistance bypass / apoptotic genes
target_genes = ['CDK7', 'CDK9', 'CDK12', 'CDK13', 'CCNT1', 'BCL2', 'BCL2L1', 'MCL1','PLK1']

matched_cols = {}
for gene in target_genes:
    matches = [c for c in header_df.columns if c.startswith(f"{gene} ")]
    if matches:
        matched_cols[gene] = matches[0]

print(f"Matched {len(matched_cols)}/{len(target_genes)} candidate resistance genes in CRISPR matrix:")
for k, v in matched_cols.items():
    print(f"  {k} -> {v}")

# 2. Extract only these target dependencies
usecols = [first_col] + list(matched_cols.values())
gene_dep_df = pd.read_csv(crispr_path, usecols=usecols)
gene_dep_df = gene_dep_df.rename(columns={first_col: 'ModelID'})

# Clean column headers
rename_dict = {v: f"{k}_Dep" for k, v in matched_cols.items()}
gene_dep_df = gene_dep_df.rename(columns=rename_dict)

# 3. Merge with our Sensitive vs Resistant cohorts
analysis_df = pd.merge(df, gene_dep_df, on='ModelID', how='inner')
analysis_df = analysis_df[analysis_df['Response_Group'].isin(['Sensitive', 'Resistant'])].copy()

print(f"\nCohort cell lines ready for testing: {len(analysis_df)}")
analysis_df.head()

Matched 9/9 candidate resistance genes in CRISPR matrix:
  CDK7 -> CDK7 (1022)
  CDK9 -> CDK9 (1025)
  CDK12 -> CDK12 (51755)
  CDK13 -> CDK13 (8621)
  CCNT1 -> CCNT1 (904)
  BCL2 -> BCL2 (596)
  BCL2L1 -> BCL2L1 (598)
  MCL1 -> MCL1 (4170)
  PLK1 -> PLK1 (5347)

Cohort cell lines ready for testing: 884


,ModelID,CDK7_Chronos_Score,lineage,primary_disease,Response_Group,MCL1_Dep,CCNT1_Dep,PLK1_Dep,CDK12_Dep,BCL2_Dep,BCL2L1_Dep,CDK7_Dep,CDK13_Dep,CDK9_Dep
1,ACH-001408,0.623595,urinary_tract,Bladder Cancer,Resistant,1.052639,1.021733,1.014616,1.043871,0.616741,1.353045,0.623595,1.046456,1.030031
2,ACH-000617,0.797398,ovary,Ovarian Cancer,Resistant,1.627950,1.101149,1.261841,0.768664,0.460995,1.357698,0.797398,1.324983,0.771605
6,ACH-000544,1.091579,esophagus,Esophageal Cancer,Sensitive,1.185638,0.976256,1.166316,0.983699,0.736724,1.259533,1.091579,1.007878,1.200436
7,ACH-001214,0.544645,central_nervous_system,Brain Cancer,Resistant,1.040400,0.999250,1.074006,0.802112,1.109998,1.438421,0.544645,1.135468,0.834226
9,ACH-000713,0.745935,ovary,Ovarian Cancer,Resistant,1.144619,1.101052,0.694189,0.648617,0.690430,1.133227,0.745935,1.636595,0.669951


In [7]:
results = []
sens_df = analysis_df[analysis_df['Response_Group'] == 'Sensitive']
res_df = analysis_df[analysis_df['Response_Group'] == 'Resistant']

for gene in matched_cols.keys():
    col = f"{gene}_Dep"
    sens_vals = sens_df[col].dropna().values
    res_vals = res_df[col].dropna().values
    
    # Mann-Whitney U test (two-sided)
    stat, pval = stats.mannwhitneyu(res_vals, sens_vals, alternative='two-sided')
    
    # Delta dependency: Positive means resistant lines rely MORE on this gene than sensitive lines
    delta_dep = np.median(res_vals) - np.median(sens_vals)
    
    # Spearman correlation with continuous CDK7 dependency score
    valid = analysis_df[['CDK7_Chronos_Score', col]].dropna()
    corr, _ = stats.spearmanr(valid['CDK7_Chronos_Score'], valid[col])
    
    results.append({
        'Gene': gene,
        'Median_Dep_in_Resistant': np.median(res_vals),
        'Median_Dep_in_Sensitive': np.median(sens_vals),
        'Delta_Dependency (Res - Sens)': delta_dep,
        'MannWhitney_P': pval,
        'Spearman_R_with_CDK7': corr
    })

results_df = pd.DataFrame(results).sort_values('MannWhitney_P').reset_index(drop=True)
results_df

,Gene,Median_Dep_in_Resistant,Median_Dep_in_Sensitive,Delta_Dependency (Res - Sens),MannWhitney_P,Spearman_R_with_CDK7
0,CDK7,0.826657,1.118725,-0.292068,4.856300e-146,1.000000
1,MCL1,1.120323,1.076136,0.044187,1.273655e-03,-0.100392
2,CCNT1,1.015320,1.037830,-0.022510,2.434410e-03,0.063006
3,BCL2L1,1.207249,1.179193,0.028057,9.390655e-02,-0.015245
4,CDK12,1.033971,1.052214,-0.018243,1.526787e-01,0.037395
5,PLK1,0.985753,1.004966,-0.019213,1.966925e-01,0.030684
6,CDK9,1.015444,1.032611,-0.017168,2.895100e-01,0.064540
7,BCL2,0.858883,0.850195,0.008687,3.536812e-01,-0.010461
8,CDK13,1.156989,1.155108,0.001881,9.122084e-01,0.050416


In [8]:
expr_path = "../data/raw/OmicsExpressionProteinCodingGenesTPMLogp1.csv"
expr_header = pd.read_csv(expr_path, nrows=0)
expr_first_col = expr_header.columns[0]

# List of hypothesized resistance & target genes
target_symbols = [
    'CDK7', 'MYC', 'MCL1', 'BCL2', 'BCL2L1', 
    'CDK9', 'CDK12', 'CDK13', 'CCNT1', 
    'ZEB1', 'VIM', 'CDH1', 'ABCB1', 'ABCG2'
]

# Map symbols to DepMap column headers ("SYMBOL (ENTREZ_ID)")
matched_cols = {}
for sym in target_symbols:
    matches = [c for c in expr_header.columns if c.startswith(f"{sym} ")]
    if matches:
        matched_cols[sym] = matches[0]

print(f"Matched {len(matched_cols)}/{len(target_symbols)} candidate biomarker columns.")

# Load only the target columns (fast & lightweight)
usecols = [expr_first_col] + list(matched_cols.values())
expr_df = pd.read_csv(expr_path, usecols=usecols)
expr_df = expr_df.rename(columns={expr_first_col: 'ModelID'})

# Rename columns back to clean gene symbols
rename_dict = {v: k for k, v in matched_cols.items()}
expr_df = expr_df.rename(columns=rename_dict)

# Merge expression with our Sensitive vs Resistant cohorts
analysis_df = pd.merge(df, expr_df, on='ModelID', how='inner')
analysis_df = analysis_df[analysis_df['Response_Group'].isin(['Sensitive', 'Resistant'])].copy()

print(f"Cell lines with matched CRISPR + Expression data: {len(analysis_df)}")
analysis_df.head()

Matched 14/14 candidate biomarker columns.
Cell lines with matched CRISPR + Expression data: 716


,ModelID,CDK7_Chronos_Score,lineage,primary_disease,Response_Group,VIM,CDH1,CDK13,ABCB1,ABCG2,CCNT1,CDK7,CDK9,MYC,MCL1,ZEB1,CDK12,BCL2L1,BCL2
1,ACH-001408,0.623595,urinary_tract,Bladder Cancer,Resistant,0.910733,7.735658,4.856488,2.003602,1.427606,3.718088,5.060480,4.449561,6.330379,6.844361,0.028569,4.218781,6.982994,1.157044
2,ACH-000617,0.797398,ovary,Ovarian Cancer,Resistant,6.926830,5.557042,4.311067,0.137504,0.238787,3.090853,4.752213,4.452859,5.575010,6.986980,0.070389,3.100978,6.630668,0.389567
5,ACH-000544,1.091579,esophagus,Esophageal Cancer,Sensitive,4.942045,4.989139,2.950468,0.321928,0.903038,2.066950,5.040454,4.459432,4.549053,5.604368,0.084064,2.871844,4.534809,0.526069
6,ACH-000713,0.745935,ovary,Ovarian Cancer,Resistant,6.429114,5.821710,4.792855,0.871844,1.432959,3.985500,5.168321,4.676944,5.021924,6.492494,0.887525,3.307429,6.558574,0.594549
7,ACH-000541,0.914121,plasma_cell,Myeloma,Resistant,9.067515,0.150560,4.547820,2.070389,0.250962,4.541019,3.981853,5.794156,9.190862,7.294069,3.351911,4.595742,6.624101,2.863938


In [9]:
stat_results = []
sens_df = analysis_df[analysis_df['Response_Group'] == 'Sensitive']
res_df = analysis_df[analysis_df['Response_Group'] == 'Resistant']

for gene in matched_cols.keys():
    sens_vals = sens_df[gene].dropna().values
    res_vals = res_df[gene].dropna().values
    
    # Non-parametric two-sided test
    stat, pval = stats.mannwhitneyu(res_vals, sens_vals, alternative='two-sided')
    
    # Delta: Positive means higher expression in Resistant lines
    delta_expr = np.median(res_vals) - np.median(sens_vals)
    
    # Continuous correlation across entire dataset
    valid = analysis_df[['CDK7_Chronos_Score', gene]].dropna()
    corr, corr_p = stats.spearmanr(valid['CDK7_Chronos_Score'], valid[gene])
    
    stat_results.append({
        'Biomarker': gene,
        'Median_Resistant': np.median(res_vals),
        'Median_Sensitive': np.median(sens_vals),
        'Delta_Expr (Res - Sens)': delta_expr,
        'MannWhitney_P': pval,
        'Spearman_R_with_Dependency': corr
    })

results_df = pd.DataFrame(stat_results).sort_values('MannWhitney_P').reset_index(drop=True)

# Benjamini-Hochberg FDR correction
n_tests = len(results_df)
results_df['rank'] = np.arange(1, n_tests + 1)
results_df['FDR_q'] = (results_df['MannWhitney_P'] * n_tests / results_df['rank']).clip(upper=1.0)
results_df['FDR_q'] = np.minimum.accumulate(results_df['FDR_q'].values[::-1])[::-1]
results_df['Significant'] = results_df['FDR_q'] < 0.05
results_df = results_df.drop(columns=['rank'])

results_df

,Biomarker,Median_Resistant,Median_Sensitive,Delta_Expr (Res - Sens),MannWhitney_P,Spearman_R_with_Dependency,FDR_q,Significant
0,CDK7,5.007196,5.736064,-0.728868,7.993375e-48,0.584507,1.119072e-46,True
1,VIM,6.603033,9.401221,-2.798188,1.264468e-09,0.215890,8.851273e-09,True
2,ZEB1,1.395063,2.720278,-1.325216,3.037169e-08,0.195081,1.417346e-07,True
3,CDH1,5.356144,1.664483,3.691661,4.477774e-07,-0.189935,1.567221e-06,True
4,ABCG2,0.765535,0.925999,-0.160465,1.884857e-02,0.080516,5.277599e-02,False
5,CDK13,4.464015,4.362470,0.101545,3.787299e-02,-0.073577,8.837031e-02,False
6,MCL1,6.563921,6.703211,-0.139291,5.235562e-02,0.038166,1.047112e-01,False
7,ABCB1,0.226509,0.310340,-0.083832,2.803426e-01,0.012966,4.905995e-01,False
8,BCL2L1,6.108943,6.268097,-0.159155,3.212924e-01,0.000593,4.997882e-01,False
9,BCL2,1.084064,1.117695,-0.033631,3.893182e-01,0.007034,5.124482e-01,False


In [10]:
# Inspect candidate bypass genes elevated in the resistant cohort
resistant_enriched = results_df.sort_values('Delta_Expr (Res - Sens)', ascending=False)
resistant_enriched[['Biomarker', 'Median_Resistant', 'Median_Sensitive', 'Delta_Expr (Res - Sens)', 'MannWhitney_P', 'FDR_q', 'Significant']]

,Biomarker,Median_Resistant,Median_Sensitive,Delta_Expr (Res - Sens),MannWhitney_P,FDR_q,Significant
3,CDH1,5.356144,1.664483,3.691661,4.477774e-07,1.567221e-06,True
5,CDK13,4.464015,4.362470,0.101545,3.787299e-02,8.837031e-02,False
10,MYC,6.121015,6.090218,0.030797,4.245121e-01,5.124482e-01,False
12,CDK9,5.190615,5.173527,0.017088,8.060970e-01,8.681044e-01,False
11,CDK12,4.331992,4.321207,0.010785,4.392413e-01,5.124482e-01,False
13,CCNT1,3.613532,3.625270,-0.011739,9.730159e-01,9.730159e-01,False
9,BCL2,1.084064,1.117695,-0.033631,3.893182e-01,5.124482e-01,False
7,ABCB1,0.226509,0.310340,-0.083832,2.803426e-01,4.905995e-01,False
6,MCL1,6.563921,6.703211,-0.139291,5.235562e-02,1.047112e-01,False
8,BCL2L1,6.108943,6.268097,-0.159155,3.212924e-01,4.997882e-01,False


In [11]:
import os

# Create repository directory for processed output tables
os.makedirs("../data/processed", exist_ok=True)

# 1. Save statistical biomarker table
results_df.to_csv("../data/processed/cdk7_resistance_biomarkers.csv", index=False)

# 2. Save cell-line expression and response matrix
analysis_df.to_csv("../data/processed/cdk7_cell_line_expression_matrix.csv", index=False)

print("Files saved directly to repository folder: data/processed/")

Files saved directly to repository folder: data/processed/
